# 02 — Les fichiers d'un sujet : `orig/`, `pre/`, vérité terrain

Ce notebook montre **chaque fichier** d'un sujet, ce qui le distingue des autres et comment ils ont été produits les uns à partir des autres. Il lit la structure BIDS produite à l'étape 2.2 (`data/bids/`).

| Section | Contenu |
|---|---|
| 1 | Inventaire : en-têtes, types, grilles, métadonnées BIDS |
| 2 | Tous les fichiers au même point physique, axes en mm |
| 3 | Le défacement de la T1 et l'effet du masque |
| 4 | Le recalage T1 → FLAIR : ce que contient le fichier elastix, reproduction exacte |
| 5 | Ce que devient la résolution quand on passe de 1 mm à 3 mm |
| 6 | Le champ de biais et sa correction (FLAIR et T1) |
| 7 | La vérité terrain sur les différentes images |

Changer de sujet : modifier `SUB` ci-dessous (`sub-000` à `sub-169`, voir `data/bids/participants.tsv`).

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import SimpleITK as sitk
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter1d
from skimage.filters import threshold_otsu

from wmh_multisite.config import load_config, resolve_path
from wmh_multisite.utils import io

SUB = "sub-000"   # Utrecht 0

cfg = load_config()
BIDS = resolve_path(cfg, "bids")
anat = BIDS / SUB / "anat"
ch = BIDS / "derivatives" / "challenge" / SUB / "anat"
man = BIDS / "derivatives" / "manual" / SUB / "anat"
FILES = {   # nom dans le challenge -> fichier BIDS
    "orig/FLAIR": anat / f"{SUB}_FLAIR.nii.gz",
    "orig/3DT1": anat / f"{SUB}_T1w.nii.gz",
    "orig/3DT1_mask": ch / f"{SUB}_space-T1w_desc-deface_mask.nii.gz",
    "orig/T1": ch / f"{SUB}_space-FLAIR_T1w.nii.gz",
    "pre/FLAIR": ch / f"{SUB}_desc-spm12_FLAIR.nii.gz",
    "pre/T1": ch / f"{SUB}_space-FLAIR_desc-spm12_T1w.nii.gz",
    "wmh (O1)": man / f"{SUB}_space-FLAIR_desc-O1_dseg.nii.gz",
}
XFM = ch / f"{SUB}_from-T1w_to-FLAIR_mode-image_xfm.txt"   # orig/reg_3DT1_to_FLAIR.txt
IMG = {k: io.load(v) for k, v in FILES.items()}
participants = pd.read_csv(BIDS / "participants.tsv", sep="\t")
participants[participants.participant_id == SUB]

## 1. Inventaire

Pour chaque fichier : géométrie (`io.describe`), est-il **sur la grille de la FLAIR** (`io.same_grid`), taille sur disque. Deux grilles seulement existent : celle de la T1 3D et celle de la FLAIR.

In [ ]:
rows = {}
for name, img in IMG.items():
    d = io.describe(img, with_stats=True)
    rows[name] = {"shape": d["shape"], "voxel (mm)": d["voxel_size_mm"], "axcodes": d["axcodes"],
                  "type disque": d["dtype_on_disk"], "grille FLAIR": io.same_grid(img, IMG["orig/FLAIR"]),
                  "p50": round(d["p50"], 1), "p99": round(d["p99"], 1), "max": round(d["max"], 1),
                  "Mo disque": round(FILES[name].stat().st_size / 1e6, 1)}
pd.DataFrame(rows).T

Les métadonnées d'acquisition écrites à la conversion BIDS (sidecars JSON, temps en secondes) :

In [ ]:
for mod in ("FLAIR", "T1w"):
    print(mod, json.dumps(json.loads((anat / f"{SUB}_{mod}.json").read_text()), indent=1, ensure_ascii=False), "\n")

## 2. Tous les fichiers au même point physique

Outil d'affichage : pour chaque image, les trois plans qui passent par **un même point en mm** (repère RAS). Les axes sont gradués en mm à partir du coin du volume, avec le sens anatomique (G → D : de la gauche vers la droite du patient, P → A, I → S). L'orientation canonique (`as_closest_canonical`) ne sert qu'à l'affichage : elle permute et retourne les axes sans interpoler.

In [ ]:
LABEL_CMAP = ListedColormap(["black", "red", "deepskyblue"])
PLANES = {"axial": ((0, 1), ("G → D (mm)", "P → A (mm)")),
          "coronal": ((0, 2), ("G → D (mm)", "I → S (mm)")),
          "sagittal": ((1, 2), ("P → A (mm)", "I → S (mm)"))}

def plane(img, plane_name, xyz, labels=False):
    can = nib.as_closest_canonical(img)
    data = io.load_labels(can) if labels else io.load_data(can)
    ijk = np.rint(np.linalg.inv(can.affine) @ np.append(xyz, 1))[:3].astype(int)
    ijk = np.clip(ijk, 0, np.array(data.shape) - 1)
    zooms = can.header.get_zooms()[:3]
    (a, b), _ = PLANES[plane_name]
    sl = {"axial": data[:, :, ijk[2]], "coronal": data[:, ijk[1], :], "sagittal": data[ijk[0], :, :]}[plane_name]
    extent = (0, data.shape[a] * zooms[a], 0, data.shape[b] * zooms[b])
    return sl.T, extent

def show(rows, xyz, planes=("axial", "coronal", "sagittal"), size=3.4):
    # rows: list of (title, image, kwargs); kwargs may contain overlay=label image, cmap, vmin, vmax
    fig, axes = plt.subplots(len(rows), len(planes), figsize=(size * len(planes), size * len(rows)), squeeze=False)
    for r, (title, img, kw) in enumerate(rows):
        for c, p in enumerate(planes):
            ax = axes[r, c]
            sl, ext = plane(img, p, xyz)
            vmax = kw.get("vmax", np.percentile(sl[sl > 0], 99.5) if (sl > 0).any() else 1)
            disp = np.ma.masked_where(sl == 0, sl) if kw.get("mask_zero") else sl   # fond transparent
            im = ax.imshow(disp, origin="lower", extent=ext, cmap=kw.get("cmap", "gray"), vmin=kw.get("vmin", 0), vmax=vmax)
            if "overlay" in kw:
                ov, _ = plane(kw["overlay"], p, xyz, labels=True)
                ax.imshow(np.ma.masked_where(ov == 0, ov), origin="lower", extent=ext, cmap=kw.get("overlay_cmap", LABEL_CMAP),
                          vmin=0, vmax=2, alpha=0.6, interpolation="nearest")
            ax.set_xlabel(PLANES[p][1][0], fontsize=7); ax.set_ylabel(PLANES[p][1][1], fontsize=7)
            ax.tick_params(labelsize=6)
            ax.set_title(f"{title} — {p}", fontsize=8)
            if kw.get("colorbar"):
                fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

flair = IMG["orig/FLAIR"]
XYZ = (flair.affine @ np.append((np.array(flair.shape) - 1) / 2, 1))[:3]   # centre de la FLAIR, en mm (RAS)
print("point affiché (mm, RAS) :", XYZ.round(1))
show([(k, IMG[k], {}) for k in ["orig/FLAIR", "pre/FLAIR", "orig/3DT1", "orig/T1", "pre/T1"]], XYZ)

**À observer**
- `orig/3DT1` est la seule image fine dans tous les plans (1 mm partout). Toutes les autres sont en coupes de 3 mm : leurs vues coronale et sagittale sont en « marches d'escalier ».
- Les coupes **axiales** de la FLAIR et de la T1 3D ne montrent pas le même niveau : les coupes FLAIR sont inclinées d'environ 25° (notebook 01). `orig/T1`, rééchantillonnée dans la grille FLAIR, montre en revanche **exactement** les mêmes plans que la FLAIR.
- `pre/*` ressemblent à `orig/*` : la correction de biais est discrète à l'œil (section 6 pour la rendre visible).

## 3. Le défacement de la T1 et l'effet du masque

Un visage reconstruit en 3D à partir d'une T1 permet d'identifier une personne : les organisateurs ont mis à zéro la région du visage. `orig/3DT1_mask` vaut 1 dans la région **conservée** et 0 dans la région **effacée** : ce n'est **pas** un masque du cerveau.

In [ ]:
t1 = io.load_data(IMG["orig/3DT1"])
mask = io.load_labels(IMG["orig/3DT1_mask"], allowed=(0, 1))
print(f"masque : {mask.mean():.0%} du volume conservé, {int((mask == 0).sum()):,} voxels effacés")
print("tous les voxels effacés valent 0 dans la T1 :", bool((t1[mask == 0] == 0).all()))

removed = nib.Nifti1Image((1 - mask).astype(np.uint8), IMG["orig/3DT1"].affine)   # 1 = effacé (affiché en rouge)
t1_center = (IMG["orig/3DT1"].affine @ np.append((np.array(t1.shape) - 1) / 2, 1))[:3]
show([("3DT1 + zone effacée (rouge)", IMG["orig/3DT1"], {"overlay": removed})], t1_center, planes=("sagittal", "axial"))

**Pourquoi ce masque compte pour le contrôle qualité.** Plusieurs métriques de qualité (SNR, EFC…) estiment le **bruit** à partir de l'arrière-plan : la zone d'air autour de la tête. Dans la zone effacée, les voxels valent exactement 0, sans bruit : les inclure ferait croire à une image beaucoup moins bruitée qu'elle ne l'est.

In [ ]:
background = t1 < threshold_otsu(t1)          # « air » grossier : tout ce qui est sous le seuil d'Otsu
for label, sel in [("arrière-plan, zone effacée incluse", background),
                   ("arrière-plan, zone effacée exclue", background & (mask == 1))]:
    v = t1[sel]
    print(f"{label:38s}: {v.size:9,d} voxels | écart-type du bruit = {v.std():6.2f} | part de zéros exacts = {(v == 0).mean():.0%}")

plt.figure(figsize=(7, 3))
plt.hist(t1[background & (mask == 1)], bins=100, alpha=0.7, label="air réel (conservé)")
plt.hist(t1[mask == 0], bins=100, alpha=0.7, label="zone effacée")
plt.yscale("log"); plt.xlabel("intensité T1"); plt.legend(); plt.title("Arrière-plan de la T1"); plt.show()

Exclure la zone effacée change l'estimation du bruit d'environ 15 % sur ce sujet. Remarquer aussi que **même l'air réel contient beaucoup de zéros exacts** : le scanner met lui-même une partie du fond à zéro. Une métrique de bruit robuste devra donc ignorer à la fois la zone défacée et les zéros du constructeur : c'est un point à traiter dans `qc/iqm.py` (Phase 4).

## 4. Le recalage T1 → FLAIR : que contient le fichier elastix ?

`orig/reg_3DT1_to_FLAIR.txt` (ici `..._from-T1w_to-FLAIR_mode-image_xfm.txt`) contient deux choses distinctes :
1. **La transformation** (`TransformParameters`, `CenterOfRotationPoint`) : 3 rotations (radians) et 3 translations (mm), en repère LPS. Elle corrige le **mouvement de la tête** entre les deux acquisitions.
2. **La grille de sortie** (`Size`, `Spacing`, `Origin`, `Direction`) : exactement celle de la FLAIR. Elle dit **où** calculer l'image résultat.

Le changement de grille lui-même (1 mm vers 0,96 × 0,96 × 3 mm, orientation PIR vers LPS) n'est pas une « transformation » : il est porté par les **en-têtes** des deux images (leurs affines), et réalisé par le rééchantillonnage.

In [ ]:
txt = XFM.read_text()
def elastix(name):
    return [float(v) for v in re.search(rf"\({name} ([^)]*)\)", txt).group(1).split()]

params, center = elastix("TransformParameters"), elastix("CenterOfRotationPoint")
print("type :", re.search(r'\(Transform "(\w+)"\)', txt).group(1))
print("rotations (°)     :", np.degrees(params[:3]).round(3))
print("translations (mm) :", np.round(params[3:], 3), "(repère LPS)")
print("centre (mm, LPS)  :", np.round(center, 2), "-> en RAS :", np.round(np.array(center) * [-1, -1, 1], 2), "= centre de la FLAIR")

fl_itk = sitk.ReadImage(str(FILES["orig/FLAIR"]))
print("\ngrille de sortie elastix = grille FLAIR :",
      np.allclose(elastix("Size"), fl_itk.GetSize()), np.allclose(elastix("Spacing"), fl_itk.GetSpacing(), atol=1e-4),
      np.allclose(elastix("Origin"), fl_itk.GetOrigin(), atol=1e-3),
      np.allclose(np.reshape(elastix("Direction"), (3, 3)).T, np.reshape(fl_itk.GetDirection(), (3, 3)), atol=1e-6),
      "(elastix range la matrice de direction colonne par colonne)")

**Reproduction.** On applique nous-mêmes cette transformation à `orig/3DT1` (SimpleITK, qui travaille comme elastix en repère LPS), sur la grille FLAIR, et on compare à `orig/T1` des organisateurs. Trois essais :
- **identité** : on ne corrige aucun mouvement, seules les affines placent les images (= notebook 01, section 9) ;
- **transformation elastix** ;
- **transformation inversée** (contrôle : le sens compte).

In [ ]:
t1_itk = sitk.ReadImage(str(FILES["orig/3DT1"]), sitk.sitkFloat32)
fl_ref = sitk.ReadImage(str(FILES["orig/FLAIR"]), sitk.sitkFloat32)
target = sitk.GetArrayFromImage(sitk.ReadImage(str(FILES["orig/T1"]), sitk.sitkFloat32))

euler = sitk.Euler3DTransform()
euler.SetCenter(center); euler.SetRotation(*params[:3]); euler.SetTranslation(params[3:])
RESULTS = {}
z, y, x = target.shape
core = np.s_[z // 4: 3 * z // 4, y // 4: 3 * y // 4, x // 4: 3 * x // 4]   # cœur du volume, loin des bords
for name, tx in [("identité (en-têtes seuls)", sitk.Transform(3, sitk.sitkIdentity)),
                 ("transformation elastix", euler), ("elastix inversée", euler.GetInverse())]:
    out = sitk.GetArrayFromImage(sitk.Resample(t1_itk, fl_ref, tx, sitk.sitkLinear, 0.0, sitk.sitkFloat32))
    RESULTS[name] = out
    d = np.abs(out - target)
    print(f"{name:28s} corrélation (cœur) = {np.corrcoef(out[core].ravel(), target[core].ravel())[0, 1]:.5f}"
          f" | écart moyen = {d[core].mean():6.2f} | écart max = {d[core].max():6.0f}")

In [ ]:
k = target.shape[0] // 2   # tableau SimpleITK : axes (z, y, x) -> coupe k de la grille FLAIR
vmax = np.percentile(target, 99)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(target[k], cmap="gray", vmax=vmax); axes[0].set_title("orig/T1 (organisateurs)")
for ax, name in zip(axes[1:], RESULTS):
    ax.imshow(np.abs(RESULTS[name][k] - target[k]), cmap="magma", vmin=0, vmax=300); ax.set_title(f"|écart| : {name}", fontsize=9)
for ax in axes: ax.axis("off")
plt.suptitle("Coupe de la grille FLAIR (axes tableau)"); plt.show()

**Conclusion.** Avec la transformation elastix, la reproduction est exacte au cœur du volume (écart maximal de 1, l'arrondi en entier de `orig/T1`). Les seuls écarts sont sur les bords du champ de vue, où les deux logiciels traitent différemment les voxels qui tombent hors de la T1. Le fichier elastix + les deux en-têtes contiennent donc **toute** l'information pour passer de `orig/3DT1` à `orig/T1`. Le mouvement corrigé est petit (environ 0,6° par axe, moins d'1 mm), mais il suffit à faire chuter la corrélation de 0,99+ à 0,925 si on l'ignore.

## 5. Que devient la résolution quand on passe de 1 mm à 3 mm ?

Le rééchantillonnage **n'additionne pas** trois voxels de 1 mm pour en faire un de 3 mm. Pour chaque voxel de la nouvelle grille, il calcule la position de son **centre** dans la T1 et **interpole** la T1 en ce point (ici linéairement, à partir des 8 voxels voisins). C'est un **échantillonnage ponctuel** : ce qui se trouve entre deux centres espacés de 3 mm n'est pas utilisé. Une réduction de résolution « propre » moyennerait d'abord sur l'épaisseur de la coupe (filtrage passe-bas) pour ne rien perdre de l'information et éviter le repliement de spectre (aliasing).

Illustration en 1D, sur un profil vertical (I → S) de la T1 3D d'origine :

In [ ]:
can = nib.as_closest_canonical(IMG["orig/3DT1"])
vol = io.load_data(can)
i, j = vol.shape[0] // 2, vol.shape[1] // 2
profile = vol[i, j, :]                                     # 1 valeur par mm, de bas en haut
zmm = np.arange(profile.size) * can.header.get_zooms()[2]
point = profile[1::3]                                      # un échantillon tous les 3 mm (échantillonnage ponctuel)
block = profile[: profile.size // 3 * 3].reshape(-1, 3).mean(axis=1)   # moyenne de 3 voxels (fusion)
smooth = gaussian_filter1d(profile, sigma=3 / 2.355)[1::3]             # filtrage puis échantillonnage

plt.figure(figsize=(11, 3.5))
plt.plot(zmm, profile, lw=0.8, color="0.6", label="T1 d'origine (1 mm)")
plt.plot(zmm[1::3], point, "o-", ms=3, label="échantillonnage ponctuel tous les 3 mm (ce que fait le rééchantillonnage)")
plt.plot(zmm[1: block.size * 3: 3], block, "s-", ms=3, label="moyenne de 3 voxels (fusion)")
plt.plot(zmm[1::3], smooth, "^-", ms=3, label="filtrage gaussien puis échantillonnage")
plt.xlabel("I → S (mm)"); plt.ylabel("intensité"); plt.legend(fontsize=7); plt.title("Profil vertical au centre de la T1")
plt.xlim(zmm.max() * 0.3, zmm.max() * 0.8); plt.show()

In [ ]:
show([("orig/3DT1 (1 mm)", IMG["orig/3DT1"], {}), ("orig/T1 (grille FLAIR, 3 mm)", IMG["orig/T1"], {})], XYZ,
     planes=("coronal", "sagittal"))

## 6. Le champ de biais et sa correction

**Le phénomène.** L'intensité mesurée n'est pas uniforme pour un même tissu : elle est multipliée par un **champ lent et lisse**, plus fort près des antennes de réception et modulé par l'inhomogénéité du champ radiofréquence d'émission (plus marquée à 3 T). Modèle : `image mesurée = image vraie × biais + bruit`.

**La correction** estime ce champ lisse et divise l'image par lui. Les organisateurs ont utilisé **SPM12** (`pre/`). Notre pipeline utilisera **N4** (ANTs) à l'étape 3.1 et se comparera à SPM12.

Le rapport `orig / pre` isole le champ estimé par SPM12 (normalisé à 1 sur sa médiane).

In [ ]:
def bias_map(orig_name, pre_name):
    o, p = io.load_data(IMG[orig_name]), io.load_data(IMG[pre_name])
    fg = o > threshold_otsu(o)
    ratio = np.zeros_like(o)
    ratio[fg] = o[fg] / np.maximum(p[fg], 1e-3)
    ratio[fg] /= np.median(ratio[fg])
    print(f"{orig_name} / {pre_name} : champ entre {np.percentile(ratio[fg], 5):.2f} et {np.percentile(ratio[fg], 95):.2f} (p5-p95)")
    return nib.Nifti1Image(np.where(fg, ratio, 0).astype(np.float32), IMG[orig_name].affine)

bias_flair, bias_t1 = bias_map("orig/FLAIR", "pre/FLAIR"), bias_map("orig/T1", "pre/T1")
kw = {"cmap": "coolwarm", "vmin": 0.75, "vmax": 1.25, "colorbar": True, "mask_zero": True}
show([("champ FLAIR (orig/pre)", bias_flair, kw), ("champ T1 (orig/pre)", bias_t1, kw)], XYZ)

Le champ n'est **pas le même** pour la FLAIR et pour la T1 : la composante d'émission dépend de la séquence, et chaque image est corrigée séparément. Effet sur un profil gauche → droite à travers la substance blanche :

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.3))
for ax, (o_name, p_name) in zip(axes, [("orig/FLAIR", "pre/FLAIR"), ("orig/T1", "pre/T1")]):
    for name, style in [(o_name, "-"), (p_name, "--")]:
        sl, ext = plane(IMG[name], "axial", XYZ)
        row = sl[sl.shape[0] // 2]
        ax.plot(np.linspace(ext[0], ext[1], row.size), row, style, lw=0.9, label=name)
    ax.set_xlabel("G → D (mm)"); ax.legend(fontsize=8); ax.set_title(f"profil horizontal au centre — {o_name.split('/')[1]}")
plt.show()

## 7. La vérité terrain sur les différentes images

`wmh.nii.gz` (O1) est sur la grille de la FLAIR : on peut la superposer sans rééchantillonner à toutes les images de cette grille (`orig/FLAIR`, `pre/FLAIR`, `orig/T1`, `pre/T1`). Les HSB sont **blanches en FLAIR** et plutôt **sombres en T1**.

In [ ]:
lab = io.load_labels(IMG["wmh (O1)"])
vox_ml = float(np.prod(IMG["wmh (O1)"].header.get_zooms()[:3])) / 1000
print({name: f"{int((lab == v).sum())} voxels = {(lab == v).sum() * vox_ml:.1f} ml" for v, name in [(1, "HSB"), (2, "autre pathologie")]})
k_best = int(np.argmax((lab == 1).sum(axis=(0, 1))))
xyz_lesion = (IMG["wmh (O1)"].affine @ np.array([lab.shape[0] / 2, lab.shape[1] / 2, k_best, 1]))[:3]
show([(k, IMG[k], {"overlay": IMG["wmh (O1)"]}) for k in ["orig/FLAIR", "orig/T1"]] +
     [("orig/FLAIR (sans masque)", IMG["orig/FLAIR"], {})], xyz_lesion, planes=("axial", "coronal"))

## Récapitulatif

| Fichier | Grille | Produit à partir de | Notre usage |
|---|---|---|---|
| `orig/FLAIR` | FLAIR | acquisition | entrée du pipeline, grille d'évaluation |
| `orig/3DT1` | T1 | acquisition, visage effacé | entrée du recalage et de HD-BET |
| `orig/3DT1_mask` | T1 | défacement | exclure la zone effacée des métriques de qualité |
| `orig/T1` | FLAIR | `3DT1` + transformation elastix + grille FLAIR | référence pour contrôler notre recalage |
| `reg_3DT1_to_FLAIR.txt` | — | elastix | mouvement de tête + grille de sortie (reproduction exacte, section 4) |
| `pre/FLAIR` | FLAIR | `orig/FLAIR` corrigée du biais (SPM12) | référence pour notre N4 |
| `pre/T1` | FLAIR | `orig/T1` corrigée du biais (SPM12), arrondie en uint16 | référence pour notre N4 |
| `wmh` | FLAIR | annotation manuelle O1 | entraînement et évaluation |